# 02 Simple Neural Network

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Build a feedforward neural network with Keras/TensorFlow
- Train it on real data (MNIST digits) and see loss/accuracy
- Understand why we use a neural net instead of hand-coded rules for this task

---

## 🌍 Real life

**Where is this used?** Handwritten digit recognition is used in **postal sorting**, **form scanning**, **bank check reading**, and **captcha systems**.

**In this notebook we use** a **feedforward neural network** to **classify handwritten digits (0–9)**. We use it **instead of** hand-coded rules or simple ML (e.g. one rule per digit) **because** the network **learns features from data**; we don't have to design the features by hand.

---

**📌 Covers slide(s):** **01** — Anatomy of NN, TensorFlow, layers, fit(), loss; **23** — Keras Sequential, callbacks. *Do this notebook after those slides.*

**Before starting:** Run the imports cell below. If TensorFlow fails with a **charset_normalizer** or **md__mypyc** error, fix the environment: run in a terminal `pip install --upgrade charset-normalizer requests`, then restart the kernel. For other errors, see `DOCS/COLAB_SETUP.md`.


## Theory (short)

- A **neural network** = layers of neurons: input → hidden layer(s) → output.
- Each layer applies **weights** and an **activation function** (e.g. ReLU).
- **Training**: forward pass → compute **loss** (how wrong we are) → **backpropagation** (gradients) → **optimizer** updates weights.
- **Data flow:** input image → flatten → Dense(128) → ReLU → Dense(10) → Softmax → predicted class.
- We use a **feedforward NN** (not CNN yet) here to keep it simple; in the next unit we'll use CNNs for images to use spatial structure.

**The steps below put this theory into code.**


## 📥 Inputs & 📤 Outputs

**Inputs:** MNIST dataset (60k training + 10k test images, 28×28 grayscale digits), TensorFlow/Keras, NumPy, Matplotlib.

**Dataset:** Real — MNIST (handwritten digits). Used so the example mimics real-world digit recognition.

**Outputs:** One figure with two plots (training/validation loss and accuracy), test accuracy number, and 5 sample predictions printed as "true label → predicted label".

**Expected:** Loss should decrease and accuracy increase over epochs; test accuracy typically >0.95 after a few epochs; most sample predictions should match the true labels. If loss stays flat, try more epochs or check learning rate.


In [ ]:
# Step 1: Imports (run this first)
# Note: uses PyTorch (compatible with macOS) instead of TensorFlow/Keras
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
print(f"PyTorch {torch.__version__}")
print("✅ Imports OK.")

In [ ]:
# Step 2: Load MNIST — using a 5 000-sample subset so the notebook finishes quickly
# (Full MNIST has 60 000 samples; the concepts are identical with 5 000)
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
transform = transforms.ToTensor()  # normalises pixels to [0, 1]
full_train = datasets.MNIST('/tmp/mnist_data', train=True,  download=True, transform=transform)
full_test  = datasets.MNIST('/tmp/mnist_data', train=False, download=True, transform=transform)
# Subset for speed (5 000 train / 1 000 test)
import torch
train_data = torch.utils.data.Subset(full_train, range(5000))
test_data  = torch.utils.data.Subset(full_test,  range(1000))
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=128)
print('Training subset:', len(train_data), '| Test subset:', len(test_data))
print('Labels: 0–9 (digits)')

In [ ]:
# Step 3: ToTensor() already normalises to [0, 1]. Flatten happens inside model.
x_sample, y_sample = full_train[0]
print('Single image tensor shape:', x_sample.shape)  # (1, 28, 28)

In [ ]:
# Step 4: Build model — feedforward NN (same structure as the Keras version)
# Input: 784 pixels → Dense(128, ReLU) → Dense(64, ReLU) → Dense(10, Softmax)
model = nn.Sequential(
    nn.Flatten(),          # 1×28×28  →  784
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),     # raw logits; CrossEntropyLoss applies softmax internally
)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# Step 5: Choose loss function and optimiser
criterion = nn.CrossEntropyLoss()          # equivalent to sparse_categorical_crossentropy
optimizer = optim.Adam(model.parameters()) # same default lr=1e-3

We use **3 epochs** here so the notebook runs in ~10 min; in real projects you'd use 10–50 (or more).


In [ ]:
# Step 6: Train for 2 epochs (enough to see loss fall and accuracy rise)
history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

for epoch in range(2):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total_loss    += loss.item() * len(yb)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total         += len(yb)
    train_loss = total_loss / total
    train_acc  = total_correct / total

    model.eval()
    vl, vc, vt = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            logits = model(xb)
            vl += criterion(logits, yb).item() * len(yb)
            vc += (logits.argmax(1) == yb).sum().item()
            vt += len(yb)
    history['loss'].append(train_loss);       history['val_loss'].append(vl/vt)
    history['accuracy'].append(train_acc);    history['val_accuracy'].append(vc/vt)
    print(f'Epoch {epoch+1}: loss={train_loss:.4f}  acc={train_acc:.4f}  '
          f'val_loss={vl/vt:.4f}  val_acc={vc/vt:.4f}')

In [ ]:
# Step 7: Plot loss and accuracy  (same as the Keras version)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(history["loss"],         label="train loss")
ax1.plot(history["val_loss"],     label="val loss")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()
ax2.plot(history["accuracy"],     label="train acc")
ax2.plot(history["val_accuracy"], label="val acc")
ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Step 8: Evaluate on test set + sample predictions
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        correct += (model(xb).argmax(1) == yb).sum().item()
        total   += len(yb)
print(f"Test accuracy: {correct/total:.4f}")

# Show 5 sample predictions
xs = torch.stack([test_data[i][0] for i in range(5)])
ys = [test_data[i][1] for i in range(5)]
with torch.no_grad():
    preds = model(xs).argmax(1).tolist()
print("Sample predictions (true → predicted):")
for i in range(5):
    print(f"  {ys[i]} → {preds[i]}")

## 🌍 Real-World Worked Example — Handwritten Digit Recognition

**Industry context:** The US Postal Service has been using digit recognition since the 1980s.  
ATMs, banks, and postal systems worldwide classify millions of handwritten digits daily.

Below we train a neural network on **real** digit images (same family as MNIST, 8×8 pixels, sklearn built-in — no download needed).

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np, matplotlib.pyplot as plt

# ── Real digit images (1797 samples, 8×8 pixels) ──────────────────────────
digits = load_digits()
X = StandardScaler().fit_transform(digits.data.astype(np.float32))
y = digits.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

X_tr_t = torch.tensor(X_tr)
y_tr_t = torch.tensor(y_tr, dtype=torch.long)
X_te_t = torch.tensor(X_te)
y_te_t = torch.tensor(y_te, dtype=torch.long)

# ── Two-layer neural network ───────────────────────────────────────────────
model = nn.Sequential(
    nn.Linear(64, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10)
)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()
losses    = []

for epoch in range(200):
    model.train()
    pred = model(X_tr_t)
    loss = loss_fn(pred, y_tr_t)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())

# ── Evaluation ──────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    acc = (model(X_te_t).argmax(1) == y_te_t).float().mean().item()

print(f"Test accuracy on real handwritten digits: {acc*100:.1f}%")

# ── Plot training loss ───────────────────────────────────────────────────
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.title("Training Loss — Handwritten Digit Classifier"); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.tight_layout(); plt.show()

# ── Show a sample prediction ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap='gray')
    pred_lbl = model(torch.tensor(X_te[:5])).argmax(1)[i].item()
    ax.set_title(f"Pred: {pred_lbl}  True: {y_te[i]}")
    ax.axis('off')
plt.suptitle("Real-World: ATM / Postal Digit Recognition", fontsize=11)
plt.tight_layout(); plt.show()

---

## 🧩 Mini-exercise

**Try it (choose one):**
- Change the number of units in the first Dense layer (e.g. 128 → 64 or → 256), retrain for 2 epochs. What happens to loss and accuracy?
- Change `epochs` to 5 and run again. Does the model overfit (train accuracy much higher than test)?

*Add a new code cell below to try your change, then run. Then read the Summary.*

---

## ✅ Summary

**What you did:**
- Loaded MNIST, normalized and flattened the images.
- Built a feedforward NN (Dense layers) and trained it for 3 epochs.
- Plotted loss/accuracy and showed sample predictions.

**In real life you'd also:** use more epochs, save the model, and optionally use a CNN to keep image structure (see Unit 2).

**The main idea:** A neural network learns from data by updating weights using the loss and backpropagation; we don't hand-design the features.

**Next:** `05_backpropagation_detailed` shows how the gradients flow through the layers (how the network "learns").


## 📚 References & Further Reading

**Foundational Papers:**
- LeCun, Bengio & Hinton (2015) — [Deep Learning](https://www.nature.com/articles/nature14539), *Nature*
- Goodfellow et al. (2016) — [Deep Learning Book](https://www.deeplearningbook.org/) (free online)

**PyTorch Docs:**
- [torch.nn — Building Blocks](https://pytorch.org/docs/stable/nn.html)
- [Autograd — Automatic Differentiation](https://pytorch.org/docs/stable/autograd.html)

**State-of-the-Art (2024–2025):**
- Transformers (GPT-4, LLaMA 3) are deep networks with 70B+ parameters
- Deep learning powers real-time translation, medical diagnosis, self-driving cars